# 접근성 지표와 이용률 상관 비교
- 목적: 생활반경 내 도달가능 가맹점 수와 실제 이용률의 관계 확인
- 비교지표: 생활반경 도달가능 가맹점 수 / 정부 최근접거리 기반 접근성
- 타깃: 구·중분류별 문화누리대상자 1인당 이용건수
- 해석방향: 지표값이 클수록 접근성이 높도록 방향을 통일함

In [ ]:
import pathlib
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

BASE_PATH = pathlib.Path().resolve()

if BASE_PATH.name == "access":
    BASE_PATH = BASE_PATH.parents[1]
elif BASE_PATH.name == "notebooks":
    BASE_PATH = BASE_PATH.parent
elif BASE_PATH.name != "oracle_mnc_project" and (BASE_PATH / "oracle_mnc_project").exists():
    BASE_PATH = BASE_PATH / "oracle_mnc_project"

ACCESS_PATH = BASE_PATH / "notebooks" / "access"
ACCESS_OUTPUT_PATH = ACCESS_PATH / "OUTPUT"
H3_PATH = ACCESS_OUTPUT_PATH / "h3sfca"
SENSITIVITY_PATH = ACCESS_OUTPUT_PATH / "h3sfca_sensitivity"
PUBLIC_PATH = ACCESS_OUTPUT_PATH / "public_access_index_25km"
OUTPUT_PATH = ACCESS_OUTPUT_PATH / "correlation_comparison"
IMAGE_PATH = ACCESS_PATH / "IMAGE" / "correlation_comparison"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
IMAGE_PATH.mkdir(parents=True, exist_ok=True)

print("BASE_PATH:", BASE_PATH)
print("H3_PATH:", H3_PATH)
print("PUBLIC_PATH:", PUBLIC_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("IMAGE_PATH:", IMAGE_PATH)

## 01. 이용률 타깃 생성
- 2025년 구·중분류별 이용건수를 불러옴
- 총 이용건수 대신 문화누리대상자 1인당 이용건수를 사용함
- 구 규모와 대상자 규모 차이를 보정하기 위한 처리임

$$
UseRate_{g,c}=\frac{Use_{g,c}}{D_g}
$$

- $Use_{g,c}$: 구 $g$, 중분류 $c$의 이용건수
- $D_g$: 구 $g$의 문화누리대상자 추정 인구수

In [ ]:
usage_model = pd.read_csv(
    SENSITIVITY_PATH / "h3sfca_usage_regression_2025_model_data.csv",
    encoding="utf-8-sig"
)

usage = (
    usage_model[["시군구", "중분류", "이용건수", "구별_문화누리대상자추정인구"]]
    .drop_duplicates()
    .copy()
)

usage["이용건수"] = pd.to_numeric(usage["이용건수"], errors="coerce").fillna(0)
usage["구별_문화누리대상자추정인구"] = pd.to_numeric(
    usage["구별_문화누리대상자추정인구"],
    errors="coerce"
).fillna(0)

usage["대상자1인당_이용건수"] = np.where(
    usage["구별_문화누리대상자추정인구"] > 0,
    usage["이용건수"] / usage["구별_문화누리대상자추정인구"],
    np.nan
)

usage["대상자천명당_이용건수"] = usage["대상자1인당_이용건수"] * 1000

category_list = sorted(usage["중분류"].dropna().unique().tolist())

print("이용률 타깃 구조:", usage.shape)
print("시군구 수:", usage["시군구"].nunique())
print("중분류 수:", usage["중분류"].nunique())
print("중분류:", category_list)
print("이용건수 결측:", usage["이용건수"].isna().sum())
print("대상자 인구 0 이하:", (usage["구별_문화누리대상자추정인구"] <= 0).sum())
print("대상자1인당 이용건수 결측:", usage["대상자1인당_이용건수"].isna().sum())

print("\n중분류별 이용률 요약")
display(
    usage
    .groupby("중분류", as_index=False)
    .agg(
        이용건수합=("이용건수", "sum"),
        대상자천명당_이용건수_평균=("대상자천명당_이용건수", "mean"),
        대상자천명당_이용건수_중앙값=("대상자천명당_이용건수", "median"),
    )
    .sort_values("대상자천명당_이용건수_평균", ascending=False)
    .round(4)
)

## 02. 생활반경 도달가능 가맹점 수 지표
- 격자·중분류별 접근가능 가맹점 수를 사용함
- 구 단위 집계는 문화누리대상자 추정 인구수 가중평균으로 계산함
- 의미: 구 내 대상자가 평균적으로 생활반경 안에서 도달 가능한 해당 분류 가맹점 수

$$
N_{g,c}=\frac{\sum_{i\in g}D_iN_{i,c}}{\sum_{i\in g}D_i}
$$

In [ ]:
def weighted_mean(group, value_col, weight_col):
    value = pd.to_numeric(group[value_col], errors="coerce")
    weight = pd.to_numeric(group[weight_col], errors="coerce").fillna(0)
    valid = value.notna() & weight.notna()

    value = value[valid]
    weight = weight[valid]

    if len(value) == 0:
        return np.nan
    if weight.sum() > 0:
        return np.average(value, weights=weight)
    return value.mean()


sfca_grid = pd.read_csv(
    H3_PATH / "sfca_no_preference_격자_중분류_접근성.csv",
    encoding="utf-8-sig",
    usecols=["시군구", "중분류", "접근가능_가맹점수", "문화누리대상자_추정_인구수"]
)

sfca_grid = sfca_grid[sfca_grid["중분류"].isin(category_list)].copy()
sfca_grid["접근가능_가맹점수"] = pd.to_numeric(sfca_grid["접근가능_가맹점수"], errors="coerce").fillna(0)
sfca_grid["문화누리대상자_추정_인구수"] = pd.to_numeric(sfca_grid["문화누리대상자_추정_인구수"], errors="coerce").fillna(0)

reachable_store = (
    sfca_grid
    .groupby(["시군구", "중분류"], as_index=False)
    .apply(
        lambda x: pd.Series({
            "지표값": weighted_mean(x, "접근가능_가맹점수", "문화누리대상자_추정_인구수"),
            "원지표값": weighted_mean(x, "접근가능_가맹점수", "문화누리대상자_추정_인구수"),
            "대상자수": x["문화누리대상자_추정_인구수"].sum(),
            "격자수": len(x)
        }),
        include_groups=False
    )
)

reachable_store["지표명"] = "생활반경_도달가능가맹점수"
reachable_store["지표단위"] = "개_대상자가중평균"
reachable_store["지표해석"] = "높을수록_생활반경내_가맹점공급많음"

reachable_store = reachable_store[[
    "시군구", "중분류", "지표명", "지표값", "원지표값", "지표단위", "지표해석", "대상자수", "격자수"
]]

print("생활반경 도달가능 가맹점 수 지표 구조:", reachable_store.shape)
print("결측:", reachable_store["지표값"].isna().sum())
print("시군구 수:", reachable_store["시군구"].nunique())
print("중분류 수:", reachable_store["중분류"].nunique())

display(reachable_store.head())

## 03. 정부 최근접거리 기반 접근성 지표
- 정부식 최근접 시설 거리 지표를 비교 지표로 사용함
- 거리는 작을수록 접근성이 높으므로, 기존 비교분석과 동일하게 역거리 접근성으로 변환함
- 지표값이 클수록 최근접 시설 거리가 짧은 방향으로 통일함

$$
G_{g,c}=\frac{1}{1+Dist_{g,c}}
$$

In [ ]:
gov_nearest = pd.read_csv(
    PUBLIC_PATH / "공공기관식_최근접접근성_서울시군구_중분류별.csv",
    encoding="utf-8-sig"
)

gov_nearest = gov_nearest[gov_nearest["중분류"].isin(category_list)].copy()
gov_nearest["문화누리대상자_가중평균_접근거리_m"] = pd.to_numeric(
    gov_nearest["문화누리대상자_가중평균_접근거리_m"],
    errors="coerce"
)

gov_access = gov_nearest[["시군구", "중분류", "문화누리대상자_가중평균_접근거리_m", "격자수"]].copy()
gov_access["지표명"] = "정부최근접거리_역거리접근성"
gov_access["지표값"] = 1 / (1 + gov_access["문화누리대상자_가중평균_접근거리_m"])
gov_access["원지표값"] = gov_access["문화누리대상자_가중평균_접근거리_m"]
gov_access["지표단위"] = "1/(1+m)"
gov_access["지표해석"] = "높을수록_최근접거리짧음"
gov_access["대상자수"] = np.nan

gov_access = gov_access[[
    "시군구", "중분류", "지표명", "지표값", "원지표값", "지표단위", "지표해석", "대상자수", "격자수"
]]

print("정부 최근접거리 역거리 접근성 구조:", gov_access.shape)
print("결측:", gov_access["지표값"].isna().sum())
print("시군구 수:", gov_access["시군구"].nunique())
print("중분류 수:", gov_access["중분류"].nunique())

display(gov_access.head())

## 04. 상관분석용 테이블 생성
- 두 접근성 지표를 같은 구·중분류 구조로 통합함
- 타깃은 대상자 1인당 이용건수만 사용함
- 전체 상관 비교에서는 분류별 단위 차이를 줄이기 위해 지표값과 이용률을 중분류 내부 z-score로 변환함

In [ ]:
def zscore_by_group(df, group_col, value_col, new_col):
    result = df.copy()

    def zscore(series):
        series = pd.to_numeric(series, errors="coerce")
        std = series.std()
        if pd.isna(std) or std == 0:
            return series * 0
        return (series - series.mean()) / std

    result[new_col] = result.groupby(group_col)[value_col].transform(zscore)
    return result


indicator = pd.concat([reachable_store, gov_access], ignore_index=True)

compare_data = indicator.merge(
    usage[["시군구", "중분류", "이용건수", "구별_문화누리대상자추정인구", "대상자1인당_이용건수", "대상자천명당_이용건수"]],
    on=["시군구", "중분류"],
    how="left"
)

compare_data = zscore_by_group(compare_data, "중분류", "지표값", "지표값_중분류내_z")
compare_data = zscore_by_group(compare_data, "중분류", "대상자1인당_이용건수", "대상자1인당_이용건수_중분류내_z")

print("상관분석용 테이블 구조:", compare_data.shape)
print("지표 수:", compare_data["지표명"].nunique())
print("시군구 수:", compare_data["시군구"].nunique())
print("중분류 수:", compare_data["중분류"].nunique())
print("지표값 결측:", compare_data["지표값"].isna().sum())
print("이용률 결측:", compare_data["대상자1인당_이용건수"].isna().sum())
print("중복:", compare_data[["시군구", "중분류", "지표명"]].duplicated().sum())

display(compare_data.head())

compare_data.to_csv(
    OUTPUT_PATH / "correlation_comparison_gu_category_data.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", OUTPUT_PATH / "correlation_comparison_gu_category_data.csv")

## 05. 중분류별 상관분석
- 중분류별로 25개 구를 기준으로 상관계수를 계산함
- Pearson은 선형 관계를 확인함
- Spearman은 순위 관계를 확인함

$$
Corr(X_{g,c},UseRate_{g,c})
$$

In [ ]:
def safe_corr(df, x_col, y_col, method):
    temp = df[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna()

    if len(temp) < 3:
        return np.nan
    if temp[x_col].nunique() < 2 or temp[y_col].nunique() < 2:
        return np.nan

    return temp[x_col].corr(temp[y_col], method=method)


category_corr_list = []

for (indicator_name, category), temp in compare_data.groupby(["지표명", "중분류"]):
    category_corr_list.append({
        "지표명": indicator_name,
        "중분류": category,
        "n": len(temp),
        "Pearson": safe_corr(temp, "지표값", "대상자1인당_이용건수", "pearson"),
        "Spearman": safe_corr(temp, "지표값", "대상자1인당_이용건수", "spearman"),
        "지표값_평균": temp["지표값"].mean(),
        "원지표값_평균": temp["원지표값"].mean(),
        "대상자천명당_이용건수_평균": temp["대상자천명당_이용건수"].mean(),
    })

category_corr = pd.DataFrame(category_corr_list)

category_summary = (
    category_corr
    .groupby("지표명", as_index=False)
    .agg(
        중분류수=("중분류", "nunique"),
        Pearson_평균=("Pearson", "mean"),
        Pearson_중앙값=("Pearson", "median"),
        Pearson_양수분류수=("Pearson", lambda x: (x > 0).sum()),
        Spearman_평균=("Spearman", "mean"),
        Spearman_중앙값=("Spearman", "median"),
        Spearman_양수분류수=("Spearman", lambda x: (x > 0).sum()),
    )
)

category_corr.to_csv(
    OUTPUT_PATH / "correlation_comparison_category_corr.csv",
    index=False,
    encoding="utf-8-sig"
)

category_summary.to_csv(
    OUTPUT_PATH / "correlation_comparison_category_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("중분류별 상관 저장:", OUTPUT_PATH / "correlation_comparison_category_corr.csv")
print("중분류별 요약 저장:", OUTPUT_PATH / "correlation_comparison_category_summary.csv")

print("\n지표별 상관 요약")
display(category_summary.round(4))

print("\n중분류별 상관")
display(category_corr.sort_values(["중분류", "지표명"]).round(4))

## 06. 전체 상관분석
- 전체 상관은 구·중분류 전체 레코드를 사용함
- 분류별 이용량 규모 차이를 줄이기 위해 중분류 내부 z-score 값을 사용함
- 해석은 보조적으로만 사용함

In [ ]:
overall_corr_list = []

for indicator_name, temp in compare_data.groupby("지표명"):
    overall_corr_list.append({
        "지표명": indicator_name,
        "n": len(temp),
        "Pearson_raw": safe_corr(temp, "지표값", "대상자1인당_이용건수", "pearson"),
        "Spearman_raw": safe_corr(temp, "지표값", "대상자1인당_이용건수", "spearman"),
        "Pearson_중분류내z": safe_corr(temp, "지표값_중분류내_z", "대상자1인당_이용건수_중분류내_z", "pearson"),
        "Spearman_중분류내z": safe_corr(temp, "지표값_중분류내_z", "대상자1인당_이용건수_중분류내_z", "spearman"),
    })

overall_corr = pd.DataFrame(overall_corr_list)

overall_corr.to_csv(
    OUTPUT_PATH / "correlation_comparison_overall_corr.csv",
    index=False,
    encoding="utf-8-sig"
)

print("전체 상관 저장:", OUTPUT_PATH / "correlation_comparison_overall_corr.csv")
display(overall_corr.round(4))

## 07. 상관분석 시각화
- 중분류별 Spearman 상관을 지표별로 비교함
- 0보다 크면 접근성이 높을수록 대상자 1인당 이용건수가 높은 방향임
- 0보다 작으면 접근성과 실제 이용률이 반대로 움직이는 방향임

In [ ]:
plot_data = category_corr.copy()
category_order = sorted(plot_data["중분류"].unique().tolist())
indicator_order = ["생활반경_도달가능가맹점수", "정부최근접거리_역거리접근성"]

pivot_s = plot_data.pivot(index="중분류", columns="지표명", values="Spearman").reindex(category_order)
pivot_p = plot_data.pivot(index="중분류", columns="지표명", values="Pearson").reindex(category_order)

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(category_order))
width = 0.36

ax.bar(
    x - width/2,
    pivot_s[indicator_order[0]],
    width,
    label="생활반경 도달가능 가맹점 수",
    color="#ea6b2d"
)
ax.bar(
    x + width/2,
    pivot_s[indicator_order[1]],
    width,
    label="정부 최근접거리 역거리 접근성",
    color="#8b1e16"
)

ax.axhline(0, color="#333333", linewidth=0.9)
ax.set_xticks(x)
ax.set_xticklabels(category_order, rotation=35, ha="right")
ax.set_ylabel("Spearman 상관계수")
ax.set_title("중분류별 접근성 지표와 대상자 1인당 이용건수 상관")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)

plt.savefig(IMAGE_PATH / "correlation_comparison_category_spearman.png", dpi=220, bbox_inches="tight")
plt.close()

fig, ax = plt.subplots(figsize=(10, 5.5))

ax.bar(
    x - width/2,
    pivot_p[indicator_order[0]],
    width,
    label="생활반경 도달가능 가맹점 수",
    color="#f3a33a"
)
ax.bar(
    x + width/2,
    pivot_p[indicator_order[1]],
    width,
    label="정부 최근접거리 역거리 접근성",
    color="#178f85"
)

ax.axhline(0, color="#333333", linewidth=0.9)
ax.set_xticks(x)
ax.set_xticklabels(category_order, rotation=35, ha="right")
ax.set_ylabel("Pearson 상관계수")
ax.set_title("중분류별 접근성 지표와 대상자 1인당 이용건수 상관")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)

plt.savefig(IMAGE_PATH / "correlation_comparison_category_pearson.png", dpi=220, bbox_inches="tight")
plt.close()

print("이미지 저장 완료")
print(IMAGE_PATH / "correlation_comparison_category_spearman.png")
print(IMAGE_PATH / "correlation_comparison_category_pearson.png")

## 08. 결과 해석용 요약
- 생활반경 내 도달가능 가맹점 수가 이용률과 더 연결되는지 확인함
- 정부 최근접거리 접근성이 이용률과 더 연결되는지 비교함
- 총 이용건수가 아니라 대상자 1인당 이용건수를 기준으로 해석함

In [ ]:
best_by_category = (
    category_corr
    .assign(abs_spearman=lambda x: x["Spearman"].abs())
    .sort_values(["중분류", "abs_spearman"], ascending=[True, False])
    .groupby("중분류", as_index=False)
    .head(1)
    .drop(columns="abs_spearman")
)

print("중분류별 절대 Spearman 기준 상관이 더 큰 지표")
display(best_by_category[["중분류", "지표명", "Pearson", "Spearman"]].round(4))

print("\n지표별 양의 상관 분류 수")
display(
    category_corr
    .groupby("지표명", as_index=False)
    .agg(
        Pearson_양수분류수=("Pearson", lambda x: (x > 0).sum()),
        Spearman_양수분류수=("Spearman", lambda x: (x > 0).sum()),
        Pearson_평균=("Pearson", "mean"),
        Spearman_평균=("Spearman", "mean"),
    )
    .round(4)
)

print("\n저장된 산출물")
for path in sorted(OUTPUT_PATH.glob("correlation_comparison*.csv")):
    print("-", path.name)

print("\n저장된 이미지")
for path in sorted(IMAGE_PATH.glob("correlation_comparison*.png")):
    print("-", path.name)

## 09. URL 여부별 가맹점 정보 결합
- 목적: 생활반경 내 가맹점 수를 URL 보유/미보유로 나누어 이용률과 비교함.
- 기준: URL 칼럼이 있는 서울 문화누리 가맹점 원자료만 사용함.
- 처리: 가맹점명·주소·중분류·소분류 정규화 키로 분석 가맹점과 URL 정보를 결합함.

In [ ]:
import re

NETWORK_OUTPUT_PATH = BASE_PATH / "analysis_table" / "data" / "output" / "network_competition_25km"
SEOUL_MERCHANT_PATH = BASE_PATH / "data" / "raw" / "merchants" / "source" / "mnc_seoul_offline_merchants_20260706.xlsx"

store = pd.read_parquet(NETWORK_OUTPUT_PATH / "경쟁권25km_분석가맹점.parquet")
seoul_store = store[store["서울여부"] == True].copy()

seoul_raw = pd.read_excel(SEOUL_MERCHANT_PATH, sheet_name=0)
seoul_raw = seoul_raw.iloc[1:].copy()
seoul_raw = seoul_raw.rename(columns={
    "분야": "대분류",
    "Unnamed: 4": "중분류",
    "Unnamed: 5": "소분류",
    "지역": "시도",
    "Unnamed: 12": "시군구"
})
seoul_raw = seoul_raw[seoul_raw["가맹점명"].notna()].copy()


def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    value = re.sub(r"\s+", "", value)
    return value


key_base_cols = ["가맹점명", "주소", "중분류", "소분류"]
key_cols = [col + "_key" for col in key_base_cols]

for col in key_base_cols:
    seoul_store[col + "_key"] = seoul_store[col].map(normalize_text)
    seoul_raw[col + "_key"] = seoul_raw[col].map(normalize_text)


def first_url(series):
    temp = series.dropna().astype(str).str.strip()
    temp = temp[temp != ""]
    if len(temp) == 0:
        return np.nan
    return temp.iloc[0]


raw_url = (
    seoul_raw
    .groupby(key_cols, as_index=False)
    .agg(
        URL=("URL", first_url),
        원자료중복행수=("가맹점명", "size")
    )
)

store_url = seoul_store.merge(
    raw_url,
    on=key_cols,
    how="left"
)

store_url["URL매칭여부"] = store_url["원자료중복행수"].notna()
store_url = store_url[store_url["URL매칭여부"]].copy()
store_url["URL보유"] = store_url["URL"].notna()

url_store_export = store_url[[
    "가맹점_ID", "가맹점명", "시군구", "주소", "대분류", "중분류", "소분류", "URL", "URL보유"
]].copy()
url_store_export.to_csv(
    OUTPUT_PATH / "correlation_comparison_url_store_match.csv",
    index=False,
    encoding="utf-8-sig"
)

print("분석 가맹점 전체:", store.shape)
print("서울 분석 가맹점:", seoul_store.shape)
print("서울 원자료:", seoul_raw.shape)
print("URL 매칭된 서울 가맹점:", store_url.shape)
print("URL 미매칭 서울 가맹점:", len(seoul_store) - len(store_url))
print("URL 보유 가맹점 수:", int(store_url["URL보유"].sum()))
print("URL 미보유 가맹점 수:", int((~store_url["URL보유"]).sum()))
print("URL 보유율:", round(store_url["URL보유"].mean() * 100, 2), "%")

print("\n중분류별 URL 보유 현황")
url_category_status = (
    store_url
    .groupby(["중분류", "URL보유"], as_index=False)
    .size()
    .pivot(index="중분류", columns="URL보유", values="size")
    .fillna(0)
    .rename(columns={False: "URL미보유", True: "URL보유"})
)
url_category_status["전체"] = url_category_status.sum(axis=1)
url_category_status["URL보유율"] = np.where(
    url_category_status["전체"] > 0,
    url_category_status["URL보유"] / url_category_status["전체"],
    np.nan
)
display(url_category_status.sort_values("전체", ascending=False).round(4))

## 10. URL 여부별 생활반경 가맹점 수 생성
- 목적: 격자별 생활반경 안에 있는 서울 가맹점을 URL 여부로 나누어 집계함.
- 기준: 기존 접근성 분석과 동일하게 도보 분류는 도보 pair, 광역 이용 분류는 대중교통 pair를 사용함.
- 저장: 격자 단위 중간 산출물은 용량 관리를 위해 저장하지 않음.

In [ ]:
import pyarrow.parquet as pq

WALK_CATEGORIES = ["도서", "문화체험", "음악", "영상", "체육시설", "체육용품"]
TRANSIT_CATEGORIES = ["미술", "공연", "스포츠관람", "관광지"]

walk_url_categories = sorted(set(category_list) & set(WALK_CATEGORIES))
transit_url_categories = sorted(set(category_list) & set(TRANSIT_CATEGORIES))

url_store_lookup = store_url[["가맹점_ID", "중분류", "URL보유"]].drop_duplicates().copy()


def count_url_reachable_pair(pair_path, lookup, categories, mode_name, batch_size=1_000_000):
    lookup_temp = lookup[lookup["중분류"].isin(categories)].copy()

    if lookup_temp.empty:
        return pd.DataFrame(columns=["GRID_CD", "중분류", "URL보유", "가맹점수", "접근수단"])

    frame_list = []
    total_pair_rows = 0
    matched_pair_rows = 0

    parquet_file = pq.ParquetFile(pair_path)

    for batch in parquet_file.iter_batches(columns=["GRID_CD", "가맹점_ID"], batch_size=batch_size):
        pair = batch.to_pandas()
        total_pair_rows += len(pair)

        pair = pair.merge(
            lookup_temp,
            on="가맹점_ID",
            how="inner"
        )

        if pair.empty:
            continue

        pair = pair.drop_duplicates(["GRID_CD", "가맹점_ID"])
        matched_pair_rows += len(pair)

        temp_count = (
            pair
            .groupby(["GRID_CD", "중분류", "URL보유"], as_index=False)
            .size()
            .rename(columns={"size": "가맹점수"})
        )
        frame_list.append(temp_count)

    if len(frame_list) == 0:
        result = pd.DataFrame(columns=["GRID_CD", "중분류", "URL보유", "가맹점수"])
    else:
        result = (
            pd.concat(frame_list, ignore_index=True)
            .groupby(["GRID_CD", "중분류", "URL보유"], as_index=False)["가맹점수"]
            .sum()
        )

    result["접근수단"] = mode_name

    print(f"{mode_name} pair 전체 행 수: {total_pair_rows:,}")
    print(f"{mode_name} URL 분석 대상 pair 행 수: {matched_pair_rows:,}")
    print(f"{mode_name} URL 집계 구조: {result.shape}")

    return result


walk_url_count = count_url_reachable_pair(
    NETWORK_OUTPUT_PATH / "경쟁권25km_도보접근성.parquet",
    url_store_lookup,
    walk_url_categories,
    "도보"
)

transit_url_count = count_url_reachable_pair(
    NETWORK_OUTPUT_PATH / "경쟁권25km_대중교통접근성.parquet",
    url_store_lookup,
    transit_url_categories,
    "대중교통"
)

url_count_long = pd.concat([walk_url_count, transit_url_count], ignore_index=True)

url_count_wide = (
    url_count_long
    .pivot_table(
        index=["GRID_CD", "중분류"],
        columns="URL보유",
        values="가맹점수",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

url_count_wide.columns.name = None

if True not in url_count_wide.columns:
    url_count_wide[True] = 0
if False not in url_count_wide.columns:
    url_count_wide[False] = 0

url_count_wide = url_count_wide.rename(columns={
    True: "URL보유_도달가능가맹점수",
    False: "URL미보유_도달가능가맹점수"
})

url_count_wide["URL전체_도달가능가맹점수"] = (
    url_count_wide["URL보유_도달가능가맹점수"]
    + url_count_wide["URL미보유_도달가능가맹점수"]
)

print("URL 여부별 격자-중분류 집계 구조:", url_count_wide.shape)
print("중복:", url_count_wide[["GRID_CD", "중분류"]].duplicated().sum())
display(url_count_wide.head())

## 11. URL 여부별 구-중분류 집계
- 목적: 격자별 URL 가맹점 수를 서울 구-중분류 단위로 집계함.
- 기준: 문화누리대상자 추정 인구수로 가중평균함.
- 해석: 구 주민이 평균적으로 생활반경 안에서 접하는 URL 보유/미보유 가맹점 수로 해석함.

In [ ]:
sfca_grid_for_url = pd.read_csv(
    H3_PATH / "sfca_no_preference_격자_중분류_접근성.csv",
    encoding="utf-8-sig",
    usecols=["GRID_CD", "시군구", "중분류", "문화누리대상자_추정_인구수"]
)

sfca_grid_for_url = sfca_grid_for_url[sfca_grid_for_url["중분류"].isin(category_list)].copy()
sfca_grid_for_url["문화누리대상자_추정_인구수"] = pd.to_numeric(
    sfca_grid_for_url["문화누리대상자_추정_인구수"],
    errors="coerce"
).fillna(0)

url_grid = sfca_grid_for_url.merge(
    url_count_wide,
    on=["GRID_CD", "중분류"],
    how="left"
)

url_count_cols = [
    "URL보유_도달가능가맹점수",
    "URL미보유_도달가능가맹점수",
    "URL전체_도달가능가맹점수"
]

for col in url_count_cols:
    url_grid[col] = pd.to_numeric(url_grid[col], errors="coerce").fillna(0)


def weighted_url_summary(group):
    weight = pd.to_numeric(group["문화누리대상자_추정_인구수"], errors="coerce").fillna(0)

    result = {
        "대상자수": weight.sum(),
        "격자수": len(group),
        "URL보유_가중합": (group["URL보유_도달가능가맹점수"] * weight).sum(),
        "URL미보유_가중합": (group["URL미보유_도달가능가맹점수"] * weight).sum(),
        "URL전체_가중합": (group["URL전체_도달가능가맹점수"] * weight).sum(),
    }

    if weight.sum() > 0:
        result["URL보유_도달가능가맹점수"] = np.average(group["URL보유_도달가능가맹점수"], weights=weight)
        result["URL미보유_도달가능가맹점수"] = np.average(group["URL미보유_도달가능가맹점수"], weights=weight)
        result["URL전체_도달가능가맹점수"] = np.average(group["URL전체_도달가능가맹점수"], weights=weight)
    else:
        result["URL보유_도달가능가맹점수"] = group["URL보유_도달가능가맹점수"].mean()
        result["URL미보유_도달가능가맹점수"] = group["URL미보유_도달가능가맹점수"].mean()
        result["URL전체_도달가능가맹점수"] = group["URL전체_도달가능가맹점수"].mean()

    if result["URL전체_가중합"] > 0:
        result["URL보유비율"] = result["URL보유_가중합"] / result["URL전체_가중합"]
    else:
        result["URL보유비율"] = np.nan

    return pd.Series(result)


url_gu_category = (
    url_grid
    .groupby(["시군구", "중분류"], as_index=False)
    .apply(weighted_url_summary, include_groups=False)
)

url_indicator = url_gu_category.melt(
    id_vars=["시군구", "중분류", "대상자수", "격자수", "URL보유_가중합", "URL미보유_가중합", "URL전체_가중합"],
    value_vars=[
        "URL보유_도달가능가맹점수",
        "URL미보유_도달가능가맹점수",
        "URL전체_도달가능가맹점수",
        "URL보유비율"
    ],
    var_name="지표명",
    value_name="지표값"
)

url_indicator = url_indicator.merge(
    usage[["시군구", "중분류", "이용건수", "구별_문화누리대상자추정인구", "대상자1인당_이용건수", "대상자천명당_이용건수"]],
    on=["시군구", "중분류"],
    how="left"
)

print("URL 구-중분류 원자료 구조:", url_gu_category.shape)
print("URL 상관분석용 long 구조:", url_indicator.shape)
print("지표값 결측:", url_indicator["지표값"].isna().sum())
print("이용률 결측:", url_indicator["대상자1인당_이용건수"].isna().sum())
print("중복:", url_indicator[["시군구", "중분류", "지표명"]].duplicated().sum())

display(url_gu_category.head())

url_indicator.to_csv(
    OUTPUT_PATH / "correlation_comparison_url_gu_category_data.csv",
    index=False,
    encoding="utf-8-sig"
)

## 12. URL 여부별 이용률 상관분석
- 목적: URL 보유 가맹점 수와 URL 미보유 가맹점 수 중 어떤 지표가 이용률과 더 가깝게 움직이는지 확인함.
- 방법: 중분류별로 구 단위 Pearson·Spearman 상관계수를 계산함.
- 주의: 상관분석은 관계의 방향과 강도를 보는 단계이며, 인과효과 검정은 아님.

In [ ]:
url_category_corr_list = []

for (indicator_name, category), temp in url_indicator.groupby(["지표명", "중분류"]):
    url_category_corr_list.append({
        "지표명": indicator_name,
        "중분류": category,
        "n": temp[["지표값", "대상자1인당_이용건수"]].replace([np.inf, -np.inf], np.nan).dropna().shape[0],
        "Pearson": safe_corr(temp, "지표값", "대상자1인당_이용건수", "pearson"),
        "Spearman": safe_corr(temp, "지표값", "대상자1인당_이용건수", "spearman"),
        "지표값_평균": temp["지표값"].mean(),
        "대상자천명당_이용건수_평균": temp["대상자천명당_이용건수"].mean(),
    })

url_category_corr = pd.DataFrame(url_category_corr_list)

url_category_summary = (
    url_category_corr
    .groupby("지표명", as_index=False)
    .agg(
        중분류수=("중분류", "nunique"),
        Pearson_평균=("Pearson", "mean"),
        Pearson_중앙값=("Pearson", "median"),
        Pearson_양수분류수=("Pearson", lambda x: (x > 0).sum()),
        Spearman_평균=("Spearman", "mean"),
        Spearman_중앙값=("Spearman", "median"),
        Spearman_양수분류수=("Spearman", lambda x: (x > 0).sum()),
    )
)

url_category_corr.to_csv(
    OUTPUT_PATH / "correlation_comparison_url_category_corr.csv",
    index=False,
    encoding="utf-8-sig"
)

url_category_summary.to_csv(
    OUTPUT_PATH / "correlation_comparison_url_category_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("URL 중분류별 상관 저장:", OUTPUT_PATH / "correlation_comparison_url_category_corr.csv")
print("URL 상관 요약 저장:", OUTPUT_PATH / "correlation_comparison_url_category_summary.csv")

print("\nURL 지표별 상관 요약")
display(url_category_summary.round(4))

print("\nURL 지표별·중분류별 상관")
display(url_category_corr.sort_values(["지표명", "중분류"]).round(4))

## 13. URL 여부별 상관 시각화
- 목적: URL 여부별 상관계수를 중분류별로 비교함.
- 기준: 순위 관계를 보기 위해 Spearman 상관계수를 중심으로 시각화함.
- 저장: URL 여부별 상관 그래프를 이미지 폴더에 저장함.

In [ ]:
url_plot_data = url_category_corr.copy()
url_indicator_order = [
    "URL보유_도달가능가맹점수",
    "URL미보유_도달가능가맹점수",
    "URL전체_도달가능가맹점수",
    "URL보유비율"
]

url_label_map = {
    "URL보유_도달가능가맹점수": "URL 보유 수",
    "URL미보유_도달가능가맹점수": "URL 미보유 수",
    "URL전체_도달가능가맹점수": "서울 가맹점 수",
    "URL보유비율": "URL 보유비율"
}

url_color_map = {
    "URL보유_도달가능가맹점수": "#ea6b2d",
    "URL미보유_도달가능가맹점수": "#8b1e16",
    "URL전체_도달가능가맹점수": "#f0a23a",
    "URL보유비율": "#168f86"
}

category_order = sorted(url_plot_data["중분류"].dropna().unique().tolist())
url_pivot_s = url_plot_data.pivot(index="중분류", columns="지표명", values="Spearman").reindex(category_order)
url_pivot_p = url_plot_data.pivot(index="중분류", columns="지표명", values="Pearson").reindex(category_order)

fig, ax = plt.subplots(figsize=(12, 5.8))
x = np.arange(len(category_order))
width = 0.18

for idx, indicator_name in enumerate(url_indicator_order):
    ax.bar(
        x + (idx - 1.5) * width,
        url_pivot_s[indicator_name],
        width,
        label=url_label_map[indicator_name],
        color=url_color_map[indicator_name]
    )

ax.axhline(0, color="#333333", linewidth=0.9)
ax.set_xticks(x)
ax.set_xticklabels(category_order, rotation=35, ha="right")
ax.set_ylabel("Spearman 상관계수")
ax.set_title("URL 여부별 생활반경 가맹점 수와 대상자 1인당 이용건수 상관")
ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.14))
ax.grid(axis="y", alpha=0.25)

plt.savefig(IMAGE_PATH / "correlation_comparison_url_category_spearman.png", dpi=220, bbox_inches="tight")
plt.close()

fig, ax = plt.subplots(figsize=(12, 5.8))

for idx, indicator_name in enumerate(url_indicator_order):
    ax.bar(
        x + (idx - 1.5) * width,
        url_pivot_p[indicator_name],
        width,
        label=url_label_map[indicator_name],
        color=url_color_map[indicator_name]
    )

ax.axhline(0, color="#333333", linewidth=0.9)
ax.set_xticks(x)
ax.set_xticklabels(category_order, rotation=35, ha="right")
ax.set_ylabel("Pearson 상관계수")
ax.set_title("URL 여부별 생활반경 가맹점 수와 대상자 1인당 이용건수 상관")
ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.14))
ax.grid(axis="y", alpha=0.25)

plt.savefig(IMAGE_PATH / "correlation_comparison_url_category_pearson.png", dpi=220, bbox_inches="tight")
plt.close()

print("저장된 URL 상관 이미지")
print("-", IMAGE_PATH / "correlation_comparison_url_category_spearman.png")
print("-", IMAGE_PATH / "correlation_comparison_url_category_pearson.png")

print("\n저장된 URL 상관 산출물")
for path in sorted(OUTPUT_PATH.glob("correlation_comparison_url*.csv")):
    print("-", path.name)